# 01 · CLIP Zero-Shot Classification & Image-Text Retrieval

**Hardware**: 🟢 CPU is fine (ViT-B/32 is ~600MB; first run downloads weights)

## What you will learn

1. How CLIP's dual towers encode images and text into **one shared embedding space**
2. Zero-shot classification = ranking image-text similarity — no classification head at all
3. Why prompt templates (`a photo of a {}`) meaningfully change accuracy
4. Seeing the famous **modality gap** in the embedding space with your own eyes
5. A known CLIP weakness: counting

## 30-second theory recap (see [theory.md](../index.md) §2)

CLIP trains on 400M image-text pairs: within a batch, matched (image, text) pairs get pulled together and mismatched ones pushed apart (InfoNCE loss). After training, the image tower and text tower land in the same space — so any classification task becomes "which sentence does this image resemble most?"

In [ ]:
%pip install -q torch transformers pillow matplotlib scikit-learn requests

In [ ]:
import torch
import matplotlib.pyplot as plt

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device = {device}")

## 1. Load CLIP

We use the classic `openai/clip-vit-base-patch32`: the image tower is a ViT-B/32, the text tower a 12-layer Transformer. Each tower's output passes through a projection layer into a shared 512-dim space.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"parameters: {n_params:.0f}M, embedding dim: {model.config.projection_dim}")

In [ ]:
# Grab a few COCO validation images as test material (swap in your own images freely)
import requests
from io import BytesIO
from PIL import Image

IMAGE_URLS = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "http://images.cocodataset.org/val2017/000000000285.jpg",
    "http://images.cocodataset.org/val2017/000000000139.jpg",
    "http://images.cocodataset.org/val2017/000000000785.jpg",
]

images = []
for url in IMAGE_URLS:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    images.append(Image.open(BytesIO(resp.content)).convert("RGB"))

fig, axes = plt.subplots(1, len(images), figsize=(16, 4))
for ax, img, url in zip(axes, images, IMAGE_URLS):
    ax.imshow(img)
    ax.set_title(url.split("/")[-1], fontsize=8)
    ax.axis("off")
plt.show()

## 2. Zero-shot classification

The recipe: write each candidate class as a sentence → encode with the text tower → cosine similarity against the image embedding → multiply by the temperature (`logit_scale`, learned during training, ~100) → softmax.

Note: **the model never trained a classifier for these labels** — that is what "zero-shot" means.

In [ ]:
labels = ["cat", "bear", "living room", "skier", "pizza", "motorcycle"]
texts = [f"a photo of a {l}" for l in labels]

inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    out = model(**inputs)

probs = out.logits_per_image.softmax(dim=-1).cpu()  # [n_images, n_labels]
print(f"logit_scale (temperature) = {model.logit_scale.exp().item():.1f}\n")

fig, axes = plt.subplots(1, len(images), figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.barh(labels, probs[i])
    ax.set_xlim(0, 1)
    ax.set_title(f"image {i}", fontsize=9)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Retrieval: the similarity matrix

Retrieval is classification transposed: given a sentence, find the most similar image. We visualize the full 4×6 cosine-similarity matrix — the sharper the diagonal structure, the better the alignment.

In [ ]:
img_emb = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
txt_emb = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
sim = (img_emb @ txt_emb.T).cpu()

plt.figure(figsize=(7, 4))
plt.imshow(sim, cmap="viridis")
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(labels)), labels, rotation=30)
plt.yticks(range(len(images)), [f"image {i}" for i in range(len(images))])
plt.title("Image-text cosine similarity")
plt.show()

# Note the value range: CLIP cosine similarities usually squeeze into 0.1–0.35.
# The ranking is meaningful; the absolute values are not probabilities —
# which is exactly why the temperature scales them up before softmax.

## 4. Prompt-template ablation

CLIP's training data is web alt-text — mostly full phrases, not bare words. So `"a photo of a cat"` usually beats the bare `"cat"`. The original OpenAI paper squeezes out a few more points by ensembling 80 templates.

In [ ]:
def classify(img, texts):
    inp = processor(text=texts, images=img, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        return model(**inp).logits_per_image.softmax(dim=-1).cpu()[0]

templates = {
    "bare label": "{}",
    "a photo of a {}": "a photo of a {}",
    "a blurry photo of a {}": "a blurry photo of a {}",
}

img = images[0]  # experiment on the first image
for name, tpl in templates.items():
    p = classify(img, [tpl.format(l) for l in labels])
    top = p.argmax().item()
    print(f"{name:28s} -> top1: {labels[top]:12s} p={p[top]:.3f}")

## 5. Visualizing the embedding space: the modality gap

Project image and text embeddings to 2D together. You'll see something counter-intuitive: **image points and text points occupy separate regions with a gap in between** — even matched pairs don't overlap. This is the modality gap (Liang et al. 2022), caused by the towers' naturally separated outputs at initialization plus a contrastive loss that only optimizes relative relations.

Takeaway: cross-modal comparison is only meaningful as *relative ranking* between images and texts — never treat absolute distance as semantic distance.

In [ ]:
from sklearn.decomposition import PCA

all_emb = torch.cat([img_emb, txt_emb]).cpu().numpy()
xy = PCA(n_components=2).fit_transform(all_emb)
n = len(images)

plt.figure(figsize=(7, 5))
plt.scatter(xy[:n, 0], xy[:n, 1], c="tab:blue", label="image", s=80)
plt.scatter(xy[n:, 0], xy[n:, 1], c="tab:orange", label="text", s=80, marker="^")
for i, l in enumerate(labels):
    plt.annotate(l, xy[n + i], fontsize=8)
for i in range(n):
    plt.annotate(f"img{i}", xy[i], fontsize=8)
plt.legend()
plt.title("CLIP embedding space (PCA) — note the separated image and text clusters")
plt.show()

## 6. Weakness probe: counting

Contrastive training only needs to "grab the main semantics" to tell in-batch negatives apart, so compositional information like counts and spatial relations is learned poorly. The first COCO image has two cats — does CLIP know?

In [ ]:
counting = [f"a photo of {n} cats" for n in ["one", "two", "three", "four"]]
p = classify(images[0], counting)
for t, prob in zip(counting, p):
    print(f"{t:28s} {prob:.3f}")
print("\nNear-uniform probabilities -> CLIP basically cannot count.")
print("This class of weakness is exactly what chapter 01's VLMs (visual features + LLM reasoning) fix.")

## Exercises

1. Swap `MODEL_ID` for `google/siglip2-base-patch16-224` (switch to `AutoModel`/`AutoProcessor`; SigLIP uses sigmoid rather than softmax, so the interface differs slightly) and rerun everything.
2. Build a mini photo album from 20 of your own pictures and implement "search my photos with a sentence".
3. Construct a spatial-relation pair (`a cat on the left of a dog` / `a dog on the left of a cat`) and check whether CLIP can tell them apart.

**Next stop**: [02_tokenize_everything.ipynb](../../02-text-io/notebooks/02_tokenize_everything.ipynb) — how each modality becomes tokens.